# Reading a Traceback: the skill nobody teaches

**Strand:** TOOLING — the six lessons that make you useful on day one in someone else's repository
**Lesson:** 04 of 06 · 2 contact hours

Every other lesson in this diploma shows you code that works. This one is about the ten seconds
after code stops working — the ten seconds that, in a job, happen many times a day and that nobody
ever teaches. Until Section 1a below, every traceback you have met anywhere in this diploma was
printed as text by a lesson that never actually failed. One cell below really does stop your
kernel. A traceback is not noise printed at you. It is a **structured report** written by the
interpreter, in a fixed format, containing the exact file, the exact line, and the exact call chain
that led to the failure. Most people scroll to the bottom, read six words, and start guessing.
You are going to learn to read the whole thing in about sixty seconds and be right.

Then you will learn the harder half, which is that **the line that broke is very often not the line
that was wrong** — and what to do about that: `pdb`, and bisection.

## 📚 Learning Objectives

By completing this notebook, you will be able to:

- Read a traceback **bottom-up** — exception type, message, crash site — then **top-down** for the
  call chain, and say out loud what each of the four parts is telling you
- Separate **where it broke** from **where it went wrong**, and name the single heuristic that finds
  the second one fastest
- Recognise on sight the errors a data person actually meets — `KeyError`, shape mismatch, dtype
  surprise, `SettingWithCopyWarning` — *and* the four failures that raise nothing at all
- **Carry on after the kernel halts** — resume from the next cell instead of restarting, reach the
  dead call again through `sys.last_value`, and say why the session must still end with
  *Restart and Run All*
- Drive a **post-mortem `pdb` session** and read live variable values at the moment of the crash
- **Bisect** a failure: over rows of data, and over commits with `git bisect run`
- Explain a real bug from this repository in which the traceback pointed at a line that was
  completely innocent

## 🔗 Where this fits

**Builds on:** `01_shell_and_filesystem` — you need a shell to see a traceback that is not inside
Jupyter, and to read an exit code; `02_git_for_one_and_for_two` — `git bisect` needs a history to
search; and `03_environments_and_dependencies` — half of all "impossible" tracebacks are a module
being imported from somewhere you did not expect.

**Used later in:** every notebook in the diploma, the moment one of them stops working — and in
particular `Course 05/unit2-cleaning/examples/02_missing_values_duplicates.ipynb`, which drops rows
and assigns into slices on nearly every page — which is where the *silent* failures of Section 5 come
from — and `Course 08/unit2-cnns/examples/07_training_cnn_image_datasets.ipynb`, which supplies the
real bug in Section 8.

**Leads to:** `05_measuring_before_optimising`, which is the same discipline applied to slowness
instead of failure — measure, do not guess — and `06_handing_over_a_repository`, where the traceback
someone else will read is one you are responsible for making readable.

**Needed for peer review:** you cannot judge someone else's work if you cannot run it and read what
it says when it fails. This lesson is the entry fee.

## 🎯 A real bug from this repository, in which the traceback lied

In August 2026 this repository gained a small health-check script, `tools/verify/parse_baseline.py`.
Its job is modest: open every notebook and syntax-check every code cell, so that a broken cell is
caught before a student meets it. Notebook cells can contain IPython magics (`%matplotlib inline`)
and shell escapes (`!pip install ...`), which are valid in Jupyter but are not Python, so the script
first strips them — any line starting with `%` or `!` is replaced by `pass` — and then compiles what
is left.

It reported a syntax error in `Course 08/unit2-cnns/examples/07_training_cnn_image_datasets.ipynb`:

```
SYNTAX cell 10: unmatched ')' (line 23)
```

Line 23 of that cell is this, and its brackets are perfectly balanced:

```python
             history.history["accuracy"][-1], history.history["val_accuracy"][-1]))
```

The notebook was fine. It ran. It still runs. What had happened is that **line 22** — the
continuation line of a multi-line `print`, which begins with the `%` string-formatting operator —
had been replaced by the word `pass`, because the stripper could not tell a magic from an operator.
The compiler was handed a mutilated string and dutifully reported the first place that string stopped
making sense, which was line 23.

So the report named a file that was not broken, a cell that was not broken, and a line that was not
broken. Every fact in it was true and every conclusion a reader would draw from it was false.
You will reproduce this bug from the real notebook, on real code, in Section 8.

This is the shape of the whole lesson:

| the traceback tells you | reliably? |
|---|---|
| **what** kind of failure it is | yes — the exception type is never wrong |
| **where** execution stopped | yes — the bottom frame is never wrong |
| **how** you got there | yes — the frame stack is never wrong |
| **why** it happened | **no.** That is your job. |

### Does anyone actually read these?

Barik et al. (2017) put 56 undergraduate and graduate software-engineering students in front of an
eye tracker and gave them defective Java to fix. The findings, in their words: participants **do**
read error messages; "the difficulty of reading these messages is comparable to the difficulty of
reading source code"; difficulty of reading them significantly predicts task performance; and
participants spent **13%–25% of their total task time** reading error messages.

Between a tenth and a quarter of debugging time is spent reading. Reading faster and more accurately
is therefore not a small optimisation. It is the largest single lever in the activity.

## 0. Lab setup

Everything below runs against **real data**: the 891-passenger Titanic manifest already in this
repository at `Course 04/datasets/raw/titanic.csv`. Nothing is downloaded and nothing outside a
temporary scratch directory is written.

The broken code lives in a **real `.py` file** that this cell writes into the scratch directory,
rather than in a notebook cell. That matters: a traceback from a notebook cell names a meaningless
temporary file, while a traceback from a file names a path and a line number you can actually open.
The last cell of the notebook deletes the scratch directory.

In [1]:
# WHAT: build a throwaway lab directory, write a deliberately broken three-step pipeline into it
# as a real .py file, and define the one helper this notebook uses to print real tracebacks.
# WHY: tracebacks are only readable when they name real files, and a scratch dir keeps the
# repository itself untouched.
import io
import os
import re
import shutil
import subprocess
import sys
import tempfile
import textwrap
import traceback
import warnings
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start=None):
    """Walk upwards until we see the two directories that only the diploma repo has."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "tools" / "verify").is_dir() and (candidate / "Course 04").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the diploma repository root from " + str(here))


REPO = find_repo_root()
TITANIC = REPO / "Course 04" / "datasets" / "raw" / "titanic.csv"

# The scratch directory. Created here, deleted in the last cell. Nothing else is written.
LAB = Path(tempfile.mkdtemp(prefix="traceback_lab_"))
os.environ["LAB_PY"] = sys.executable          # so the %%bash cell later uses THIS interpreter
os.environ["LAB_DIR"] = str(LAB)
os.environ["TITANIC_PATH"] = str(TITANIC)

PIPELINE_SRC = textwrap.dedent('''
    """A three-step pipeline over the real Titanic passenger manifest."""
    import pandas as pd


    def load_passengers(csv_path):
        """Step 1 - read the file exactly as it is on disk. 891 rows, 12 columns."""
        return pd.read_csv(csv_path)


    def clean_ages(df):
        """Step 2 - 177 of the 891 passengers have no recorded Age, so remove them."""
        return df["Age"].dropna()


    def age_times_fare(ages, df):
        """Step 3 - element-wise product of the two columns."""
        return ages.to_numpy() * df["Fare"].to_numpy()


    def report(csv_path):
        """The whole job: load, clean, combine, average."""
        df = load_passengers(csv_path)
        ages = clean_ages(df)
        return float(age_times_fare(ages, df).mean())
''').lstrip()
(LAB / "pipeline.py").write_text(PIPELINE_SRC, encoding="utf-8")

sys.path.insert(0, str(LAB))
sys.modules.pop("pipeline", None)              # so re-running this cell picks up the file again
import pipeline                                # noqa: E402  (import must follow the sys.path edit)

# Exact prefixes, computed rather than guessed -- the repo path contains a space, and macOS
# resolves /var to /private/var, so a regex over these would be wrong in two different ways.
SITE_PACKAGES = str(Path(pd.__file__).parents[1]) + os.sep
LAB_PREFIXES = sorted({str(Path(LAB).resolve()), str(LAB)}, key=len, reverse=True)
CELL_FILE = re.compile(r'File "[^"]*ipykernel_\d+[^"]*"')


def tidy(text):
    """Shorten only the PATHS in a traceback so it fits the page.

    Nothing else is altered: every line number, function name, message and caret
    below is exactly what Python produced.
    """
    for prefix in LAB_PREFIXES:
        text = text.replace(prefix + os.sep, "lab/")
    text = text.replace(SITE_PACKAGES, ".../site-packages/")
    return CELL_FILE.sub('File "<this notebook>"', text)


def pdb_session(setup_code, commands):
    """Run `setup_code` in a FRESH interpreter; when it raises, drop into a real
    standard-library pdb post-mortem and feed it `commands` on stdin.

    A subprocess is used on purpose: inside IPython, `pdb` is replaced by a coloured
    variant, and what you will meet in a terminal is the plain one. This transcript is
    therefore exactly what you would see after typing the same commands yourself.
    """
    driver = LAB / "pdb_driver.py"
    driver.write_text(
        "import sys, pdb\n"
        f"sys.path.insert(0, {str(LAB)!r})\n"
        "try:\n" + textwrap.indent(setup_code, "    ") + "\n"
        "except Exception:\n"
        "    try:\n        pdb.post_mortem()\n"
        "    except BaseException:\n        pass\n",
        encoding="utf-8")
    done = subprocess.run([sys.executable, "pdb_driver.py"], cwd=str(LAB),
                          input=commands, capture_output=True, text=True, timeout=120)
    return tidy(done.stdout).rstrip()


def show_traceback(fn, *args, **kwargs):
    """Run code that is expected to fail, and print its REAL traceback.

    The only frame removed is this helper's own, so that the printout starts at
    the code under study instead of at the harness. Everything else is untouched.
    """
    harness = show_traceback.__code__.co_filename
    try:
        fn(*args, **kwargs)
    except BaseException as exc:                       # noqa: BLE001 - teaching harness
        report = traceback.TracebackException.from_exception(exc)
        node = report
        while node is not None:
            node.stack = traceback.StackSummary.from_list(
                [f for f in node.stack
                 if not (f.filename == harness and f.name == "show_traceback")]
            )
            node = node.__cause__ or node.__context__
        print(tidy("".join(report.format()).rstrip()))
        return exc
    print("(no exception was raised)")
    return None


print(f"repository root : {REPO.name}")
print(f"real dataset    : {TITANIC.relative_to(REPO)}  ({TITANIC.stat().st_size / 1024:.0f} KB)")
print(f"scratch lab dir : {LAB.name}  (deleted by the last cell)")
print(f"python          : {sys.version.split()[0]}   pandas {pd.__version__}   numpy {np.__version__}")

repository root : AI Diploma
real dataset    : Course 04/datasets/raw/titanic.csv  (59 KB)
scratch lab dir : traceback_lab_b40amygd  (deleted by the last cell)
python          : 3.14.3   pandas 2.3.3   numpy 2.4.4


## 1. The anatomy of a traceback

Run the broken pipeline. Do not read the output yet — read the four rules first, then read it.

**Rule 1 — start at the bottom.** The very last line is the only line guaranteed to describe the
failure itself. It has two halves separated by a colon: the **exception type** on the left, and the
**message** on the right. The type is a fact chosen by the interpreter or the library; the message
is prose written by a human who may or may not have anticipated your situation. Trust the type more
than the message.

**Rule 2 — the frame directly above the last line is where execution stopped.** Not where the
mistake was made. Where it *stopped*.

**Rule 3 — read upwards through the frames to reconstruct the call chain.** The frames are printed
oldest-first, so reading the block from the top gives you the story in the order it happened:
`report` called `age_times_fare`, which is where it died.

**Rule 4 — find the lowest frame that is your own code.** Library frames tell you *how* the library
handled your bad input. Your lowest frame is where you handed it over. That frame is where the fix
usually goes, and it is the single most useful line in the whole printout.

### 1a. First, the one that stops

Every traceback printed later in this notebook is **caught**: a helper runs the broken code inside
`try`/`except`, formats the exception and prints it as ordinary text. That is convenient for
dissecting one on the page, and it is not what happens to you at work.

The next cell has no helper around it. It calls the broken pipeline directly, the exception escapes,
and **your kernel stops there**. If you used *Run All*, the run ends at that cell and nothing below it
executes. That is deliberate, it is the only cell in this notebook that behaves this way, and it is
the whole reason the lesson exists.

Read the red block. Then click the cell **below** it and press Shift+Enter to carry on. Do not
restart the kernel — the next section is about why that is the expensive reflex.

In [2]:
# WHAT: call the broken pipeline with nothing wrapped around it, so the exception escapes.
# WHY: a caught traceback is a photograph of a fire. This one is the fire - and it costs you what a
#      real one costs: execution stops HERE, so every cell below is unrun until you resume it.
pipeline.report(TITANIC)

ValueError: operands could not be broadcast together with shapes (714,) (891,) 

### The ten seconds after the kernel stops

Hold that block against the four rules you just read. The structure is identical — banner, frames
oldest-first, type and message at the bottom — but two things differ from every other traceback in
this notebook, and both are things you will see every working day:

**The top frame is now yours.** It reads `Cell In[...]`, and it is the line *you* typed. Rule 4 said
to find the lowest frame that is your own code; here the highest one is yours as well, because you
are the caller. The two `lab/pipeline.py` frames underneath are the same two frames the rest of the
lesson dissects.

**The paths are full length** — `/var/folders/…/traceback_lab_…/pipeline.py`, or whatever your
operating system chose. Real tracebacks look like this. Every later cell in this notebook shortens
those paths to `lab/` so they fit the page; **only** the paths. Every line number, function name,
caret and message stays exactly as Python produced it.

Now the habit, which is three moves and takes about ten seconds:

1. **Do not restart.** A halted kernel is not a dead kernel. Nothing you loaded was lost: the
   `pipeline` module is still imported, the scratch directory is still on disk, and the failed call
   itself is still reachable. The next cell proves all three. People restart out of reflex and then
   spend twenty minutes reloading a dataframe they still had.
2. **Resume from the next cell**, not from the top. You are now running cells out of order. That is
   correct while you are debugging and a liability the moment you stop.
3. **When the fix is in, Restart and Run All.** `06_handing_over_a_repository` calls this the
   ten-minute test, and it is not a formality. Samuel and Mietchen re-ran 2,684 notebooks from
   biomedical publications — notebooks whose declared dependencies had *all* installed successfully,
   so the hard part was already over. **396 ran through without any errors** and **245 produced
   results identical to those reported**: 14.8% and 9.1%. Every one of those notebooks worked in its
   author's execution order. Order is not a detail; it is most of the gap.

In [3]:
# WHAT: the three checks that turn a halt into a starting point instead of a restart.
print(f"pipeline module still imported : {'pipeline' in sys.modules}")
print(f"scratch lab still on disk      : {LAB.exists()}")

# IPython parks the whole failed call in sys.last_* after an uncaught exception, so `%debug` in a
# fresh cell opens a pdb post-mortem on it - no need to reproduce anything. Section 6 scripts the
# same session in a plain terminal, where pdb has no colours to hide behind.
print(f"sys.last_value                 : {type(sys.last_value).__name__}: {sys.last_value}")

stored = traceback.extract_tb(sys.last_traceback)
print(f"frames still on that traceback : {len(stored)}")
for depth, f in enumerate(stored, start=1):
    print(f"   {depth}  {tidy(f.filename)}  line {f.lineno}  in {f.name}()")

pipeline module still imported : True
scratch lab still on disk      : True
sys.last_value                 : ValueError: operands could not be broadcast together with shapes (714,) (891,) 
frames still on that traceback : 4
   1  .../site-packages/IPython/core/interactiveshell.py  line 3747  in run_code()
   2  /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/ipykernel_9945/468398733.py  line 4  in <module>()
   3  lab/pipeline.py  line 24  in report()
   4  lab/pipeline.py  line 17  in age_times_fare()


Nothing was lost. `sys.last_value` is the same `ValueError` object you read in red, so you can open
a debugger on the moment of the crash right now — `%debug` in a fresh cell — without reproducing
anything at all.

Then read the **four** frames it listed against the **three** the red block drew. They do not match,
and the mismatch is worth four seconds:

| # | frame the traceback object actually holds | drawn in the red block? |
|---|---|---|
| 1 | `IPython/core/interactiveshell.py`, `run_code()` | **no** — IPython hides its own machinery |
| 2 | `…/ipykernel_…/<a number>.py` line 4, `<module>()` | yes, but drawn as `Cell In[…], line 4` |
| 3 | `lab/pipeline.py` line 24, `report()` | yes |
| 4 | `lab/pipeline.py` line 17, `age_times_fare()` | yes |

**The printed traceback is a rendering, not the thing.** Someone decided which frames were worth your
attention, and that someone was right — but it means the text is an opinion about a data structure,
and the data structure is available to you. Frame 2 also shows why Section 0 wrote the broken code
into a real `.py` file: a notebook cell's own frame is named after a random number, and there is no
such file to open.

At work this pairing has a name. Code that is *supposed* to fail is never left to chance — you assert
the failure, with `pytest.raises` in a test suite, or, as this notebook does, with a
`raises-exception` tag that the repository's checker reads. **⚠️ Where this breaks** says what that
tag does and does not buy you.

From here the notebook goes back to **caught** tracebacks, for one reason: a caught traceback can be
printed, shortened, tabulated and compared, and a halting one cannot. The next cell reruns the same
failure inside that harness. Same exception, same frames 3 and 4, same line numbers — shorter paths.

In [4]:
# WHAT: rerun the SAME failure inside a harness that catches it, and print what Python emitted.
# WHY: this is the artefact the rest of the lesson dissects, and a caught one can be shortened to
#      fit the page. Only the paths differ from the red block above.
print("=" * 78)
print("THE SAME FAILURE, CAUGHT  (lab/pipeline.py, 891 real passengers, short paths)")
print("=" * 78)
failure = show_traceback(pipeline.report, TITANIC)

THE SAME FAILURE, CAUGHT  (lab/pipeline.py, 891 real passengers, short paths)
Traceback (most recent call last):
  File "lab/pipeline.py", line 24, in report
    return float(age_times_fare(ages, df).mean())
                 ~~~~~~~~~~~~~~^^^^^^^^^^
  File "lab/pipeline.py", line 17, in age_times_fare
    return ages.to_numpy() * df["Fare"].to_numpy()
           ~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~~~~
ValueError: operands could not be broadcast together with shapes (714,) (891,)


### Reading what just printed, line by line

```
Traceback (most recent call last):                 <- fixed banner. "most recent call LAST" is the
                                                      whole reading instruction: the bottom is now.

  File "lab/pipeline.py", line 24, in report       <- FRAME 1: the outer call. Line 24 is the line
    return float(age_times_fare(ages, df).mean())     inside report() that was executing.
                 ~~~~~~~~~~~~~~^^^^^^^^^^          <- carets: the exact sub-expression (Section 3)

  File "lab/pipeline.py", line 17, in age_times_fare   <- FRAME 2: the crash site. Execution stopped
    return ages.to_numpy() * df["Fare"].to_numpy()        HERE, on line 17.
           ~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~~~~    <- the ^ sits under the `*` operator

ValueError: operands could not be broadcast together with shapes (714,) (891,)
^^^^^^^^^^                                          <- TYPE: numpy is refusing an operation
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^  <- MESSAGE: and it tells you both shapes
```

Four seconds of reading has already produced a complete factual account: *two arrays of length 714
and 891 were multiplied together on line 17 of `pipeline.py`, inside `age_times_fare`, which was
called from line 24 of `report`.*

Notice what it has **not** told you: which of the two arrays is wrong. `age_times_fare` did nothing
incorrect — it multiplied the two things it was given. The next cell puts the same information into a
table so you can see the structure independently of the formatting.

In [5]:
# WHAT: walk the stored traceback object frame by frame and tabulate it.
# WHY: the printed text is one rendering of a real data structure; being able to inspect that
# structure is what lets you write your own error reporting later.
frames = [f for f in traceback.extract_tb(failure.__traceback__)
          if f.name != "show_traceback"]
rows = []
for depth, f in enumerate(frames, start=1):
    is_mine = str(LAB) in f.filename          # "my code" == the lab file, not a library
    rows.append({
        "frame": depth,
        "file": tidy(f.filename),
        "line": f.lineno,
        "function": f.name,
        "whose code?": "YOURS" if is_mine else "library",
        "source": (f.line or "").strip()[:44],
    })

print(pd.DataFrame(rows).to_string(index=False))
print()
print(f"exception type    : {type(failure).__name__}")
print(f"exception message : {failure}")
mine = [r for r in rows if r["whose code?"] == "YOURS"]
print(f"deepest frame that is YOUR code: frame {mine[-1]['frame']}, "
      f"{mine[-1]['file']} line {mine[-1]['line']} in {mine[-1]['function']}()")

 frame            file  line       function whose code?                                       source
     1 lab/pipeline.py    24         report       YOURS return float(age_times_fare(ages, df).mean()
     2 lab/pipeline.py    17 age_times_fare       YOURS return ages.to_numpy() * df["Fare"].to_numpy

exception type    : ValueError
exception message : operands could not be broadcast together with shapes (714,) (891,) 
deepest frame that is YOUR code: frame 2, lab/pipeline.py line 17 in age_times_fare()


## 2. Where it broke is not where it went wrong

The traceback says line 17. Open line 17 and it is blameless:

```python
return ages.to_numpy() * df["Fare"].to_numpy()
```

There is nothing to fix there. Multiplying two arrays of the same length is exactly the right thing
to do, and it is what the function is for. The mistake was made **five lines earlier and one frame
up**, by a decision that looked entirely reasonable at the time:

```python
def clean_ages(df):
    return df["Age"].dropna()      # <- silently changes the length of the result
```

`dropna()` is not a bug. It is a decision — *"passengers with no recorded age should be excluded"* —
and it is probably even the right decision. What makes it a defect is that it was made in a function
whose caller assumed the length was preserved. **The failure is in the contract between two
functions, and neither function contains it.**

The next cell proves the diagnosis with numbers rather than with a story: it prints the lab file with
line numbers, and then measures the three lengths involved.

In [6]:
# WHAT: print the source under the traceback's line numbers, then measure the actual lengths.
# WHY: a diagnosis is a claim about the program's state; a claim about state is settled by measuring
# it, never by re-reading the code and feeling confident.
print("lab/pipeline.py  (the line numbers here are the ones the traceback quoted)")
print("-" * 78)
for n, line in enumerate(PIPELINE_SRC.splitlines(), start=1):
    marker = " <-- traceback said this" if n in (17, 24) else ""
    print(f"{n:>3} | {line}{marker}")

print()
print("Now measure, instead of guessing:")
df_all = pipeline.load_passengers(TITANIC)
ages_clean = pipeline.clean_ages(df_all)
print(f"  rows in the file                       : {len(df_all)}")
print(f"  passengers with a recorded Age         : {len(ages_clean)}")
print(f"  passengers with Age missing (dropped)  : {df_all['Age'].isna().sum()}")
print(f"  {len(ages_clean)} + {df_all['Age'].isna().sum()} = {len(ages_clean) + df_all['Age'].isna().sum()}"
      f"  <- matches the two shapes in the ValueError exactly")

lab/pipeline.py  (the line numbers here are the ones the traceback quoted)
------------------------------------------------------------------------------
  1 | """A three-step pipeline over the real Titanic passenger manifest."""
  2 | import pandas as pd
  3 | 
  4 | 
  5 | def load_passengers(csv_path):
  6 |     """Step 1 - read the file exactly as it is on disk. 891 rows, 12 columns."""
  7 |     return pd.read_csv(csv_path)
  8 | 
  9 | 
 10 | def clean_ages(df):
 11 |     """Step 2 - 177 of the 891 passengers have no recorded Age, so remove them."""
 12 |     return df["Age"].dropna()
 13 | 
 14 | 
 15 | def age_times_fare(ages, df):
 16 |     """Step 3 - element-wise product of the two columns."""
 17 |     return ages.to_numpy() * df["Fare"].to_numpy() <-- traceback said this
 18 | 
 19 | 
 20 | def report(csv_path):
 21 |     """The whole job: load, clean, combine, average."""
 22 |     df = load_passengers(csv_path)
 23 |     ages = clean_ages(df)
 24 |     return float(age_t

The two numbers in the exception message, `(714,)` and `(891,)`, appear in the measurements above as
"passengers with a recorded Age" and "rows in the file". The traceback's message and the program's
state agree, which means the diagnosis is confirmed rather than merely plausible.

Now the fix — and note that there are **three** defensible fixes, which is the usual situation and
the reason debugging is a judgement activity rather than a lookup:

1. Drop the rows from **both** columns together (`df.dropna(subset=["Age"])`) — keeps the pairing.
2. Do not drop at all; let pandas align on the index and produce `NaN` where Age is missing.
3. Fill the missing ages with something defensible, and say so in writing.

They do not have to give the same answer, so choosing between them is a modelling decision rather
than a syntax decision — and the cell below shows exactly how treacherous that is. Two of the three
land on the same number here, for a reason that is not obvious, while a fourth line that most people
would consider the same fix lands somewhere else entirely. Read the printed output before you decide
which repair you would have committed.

In [7]:
# WHAT: implement two of the three defensible repairs and compare their answers on the real data.
# WHY: "the bug is fixed" is not one outcome. Different correct-looking repairs give different
# numbers, and the traceback has no opinion about which you wanted.
def repaired_drop_together(csv_path):
    """Fix 1: drop Age-missing rows from the WHOLE frame, so the two columns stay paired."""
    df = pd.read_csv(csv_path)
    paired = df.dropna(subset=["Age"])
    return float((paired["Age"].to_numpy() * paired["Fare"].to_numpy()).mean())


def repaired_keep_alignment(csv_path):
    """Fix 2: never drop; multiply the SERIES so pandas aligns on the index, then average."""
    df = pd.read_csv(csv_path)
    product = df["Age"] * df["Fare"]        # index-aligned; NaN wherever Age is missing
    return float(product.mean())            # Series.mean() skips NaN by default


a = repaired_drop_together(TITANIC)
b = repaired_keep_alignment(TITANIC)
print(f"fix 1  drop the rows from both columns : mean(Age x Fare) = {a:.4f}   over 714 passengers")
print(f"fix 2  keep everything, align on index : mean(Age x Fare) = {b:.4f}   over 714 non-NaN products")
print(f"difference                             : {abs(a - b):.10f}")
print()
print("They agree here because Series.mean() skips NaN, so fix 2 silently averages the same 714")
print("products. Change the last line to product.fillna(0).mean() and the answer moves to",
      f"{float((df_all['Age'] * df_all['Fare']).fillna(0).mean()):.4f} -- same 'fixed' code, different science.")

fix 1  drop the rows from both columns : mean(Age x Fare) = 1104.1421   over 714 passengers
fix 2  keep everything, align on index : mean(Age x Fare) = 1104.1421   over 714 non-NaN products
difference                             : 0.0000000000

They agree here because Series.mean() skips NaN, so fix 2 silently averages the same 714
products. Change the last line to product.fillna(0).mean() and the answer moves to 884.8007 -- same 'fixed' code, different science.


### 🔑 The heuristic that finds the real culprit fastest

> **Start at the deepest frame that belongs to you, and walk upwards asking one question of each
> frame: "was the value this frame received already wrong when it arrived?"**
>
> The first frame where the answer is *no — the value was fine here, and it was this frame that
> spoiled it* is the bug.

In the traceback above, frame 2 (`age_times_fare`) received a 714-element array and an 891-element
frame: already wrong on arrival. Frame 1 (`report`) received a path: fine on arrival, and it was
`report`'s own call to `clean_ages` that produced the mismatched length. So the fix belongs in
`report` or in `clean_ages` — and note that **`clean_ages` does not appear in the traceback at all**,
because it had already returned successfully before the crash. The function that caused the failure
is frequently not in the printout. This is why the traceback is a starting point rather than an
answer.

## 3. Two things modern Python gives you for free

**PEP 657 — *Include Fine Grained Error Locations in Tracebacks*** (Galindo Salgado, Taskaya & Askar;
Final; landed in Python 3.11) is the reason for the `~~~~^^^^` markers. Before 3.11 a traceback
could only name a line. On a line like `a["x"]["y"]["z"] = 1` that left four candidates and you had
to guess. Now the `^` characters sit under the exact sub-expression that raised, and the `~`
characters show the surrounding expression it belongs to. **Read the carets before you read the
line.**

**PEP 3134 — *Exception Chaining and Embedded Tracebacks*** (Yee; Final; Python 3.0) is the reason a
traceback sometimes contains two or three tracebacks
stacked on top of each other, joined by one of two sentences:

- `The above exception was the direct cause of the following exception:` — someone wrote
  `raise NewError(...) from old`. Deliberate. The chain is a designed translation of one error into
  a more meaningful one.
- `During handling of the above exception, another exception occurred:` — the second failure happened
  *inside an `except` block* and nobody linked them on purpose. Often the handler itself is broken.
  **This one is a smell.**

In a chain, the **topmost** traceback is the original cause and the **bottom** one is what finally
escaped. So you read a chain top-down for the cause and bottom-up for the effect — the opposite of a
single traceback, which is why chains confuse people.

**PEP 678 — *Enriching Exceptions with Notes*** (Hatfield-Dodds; Final; Python 3.11) adds
`exception.add_note("...")`, which appends a line of your own text directly under the exception
message. It is the cheapest way to put *which file / which row / which parameter* into a traceback
that would otherwise not say.

In [8]:
# WHAT: produce three real tracebacks that show carets, a deliberate chain, and an accidental chain.
# WHY: these three shapes cover almost every confusing traceback a beginner meets.
df = pd.read_csv(TITANIC)

print("A) CARETS: a KeyError inside a longer expression -- note WHERE the ^ sits")
print("-" * 78)


def average_fare_per_head(frame):
    """'Party Size' does not exist in this file; the other three sub-expressions are fine."""
    return frame["Fare"].mean() / frame["Party Size"].mean()


show_traceback(average_fare_per_head, df)

print()
print("B) DELIBERATE CHAIN: 'raise ... from e' -- the library author translating an error")
print("-" * 78)


def deck_letters(frame):
    try:
        return frame["Deck"].str[0]
    except KeyError as exc:
        raise ValueError("this manifest has no Deck column; derive it from Cabin first") from exc


_ = show_traceback(deck_letters, df)

A) CARETS: a KeyError inside a longer expression -- note WHERE the ^ sits
------------------------------------------------------------------------------
Traceback (most recent call last):
  File ".../site-packages/pandas/core/indexes/base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'Party Size'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "<this notebook>", line 11, in average_fare_per_head
    return frame["Fare"].mean() / frame["Party Size"].mean(

In [9]:
# WHAT: the accidental chain, plus PEP 678 notes attached to a real failure.
# WHY: 'During handling of the above exception' almost always means the HANDLER is broken, and
# add_note() is how you stop a traceback from being anonymous.
print("C) ACCIDENTAL CHAIN: the except block itself fails")
print("-" * 78)


def fragile_handler(frame):
    try:
        return frame["Deck"].str[0]
    except KeyError:
        # The handler tries to log something helpful and gets it wrong -- a very common shape.
        return "column missing: " + frame.shape[1]      # int cannot be concatenated to str


show_traceback(fragile_handler, df)

print()
print("D) PEP 678 NOTES: the same failure, but the traceback now says WHICH file and WHICH column")
print("-" * 78)


def annotated_deck_letters(frame, source):
    try:
        return frame["Deck"].str[0]
    except KeyError as exc:
        exc.add_note(f"source file : {source.name}")
        exc.add_note(f"columns present : {', '.join(frame.columns)}")
        exc.add_note("hint : Cabin holds strings like 'C85'; Deck would be its first character")
        raise


_ = show_traceback(annotated_deck_letters, df, TITANIC)

C) ACCIDENTAL CHAIN: the except block itself fails
------------------------------------------------------------------------------
Traceback (most recent call last):
  File ".../site-packages/pandas/core/indexes/base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'Deck'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "<this notebook>", line 10, in fragile_handler
    return frame["Deck"].str[0]
           ~~~~~^^^^^^^^
  File ".../site-packages/pandas/core

Compare **C** with **B**. In C the last exception is a `TypeError` about string concatenation, and a
reader who scrolls to the bottom will spend their afternoon on string concatenation. The real problem
is in the *first* traceback of the chain: there is no `Deck` column. The sentence *"During handling
of the above exception, another exception occurred"* is the interpreter telling you, politely, that
the bottom half is a distraction.

Compare **D** with **A**. Same underlying failure, same exception type — but D's traceback carries
the source filename and the list of columns that *do* exist, so it can be diagnosed without opening
a single file. Three lines of `add_note` turned a question into an answer. When you write code other
people will run, this is the highest-value habit in this notebook.

## 4. The loud six: errors a data person meets every week

The cell below runs six genuinely broken snippets against the real manifest, catches each one, and
prints the last line — the type and message. All six are produced live; none is a quotation.

Read the table afterwards as a **lookup for the first thing to check**, not as an explanation. The
message tells you what the interpreter noticed. The first-thing-to-check column is what experience
says is usually behind it.

In [10]:
# WHAT: raise six real, distinct exceptions against the real Titanic file and collect their last lines.
# WHY: recognising the SHAPE of an error on sight is most of the speed; the content varies, the
# shape does not.
df = pd.read_csv(TITANIC)
whitespace_df = df.rename(columns={"Embarked": "Embarked "})     # a real CSV-export accident


def e_keyerror():
    return df["Ticket Number"]                       # column simply does not exist


def e_keyerror_whitespace():
    return whitespace_df["Embarked"]                 # column exists, name has a trailing space


def e_shape():
    return df["Age"].dropna().to_numpy() * df["Fare"].to_numpy()


def e_dtype():
    return df["Name"] + df["Fare"]                   # str + float, element-wise


def e_nan_is_a_float():
    return [port[0] for port in df["Embarked"]]      # 2 of the 891 values are NaN, i.e. a float


def e_indexerror():
    return df["Fare"].to_numpy()[len(df)]            # classic off-by-one at the end of an array


cases = [
    ("KeyError (missing)", e_keyerror, "print list(df.columns) and diff it against what you typed"),
    ("KeyError (whitespace)", e_keyerror_whitespace, "print [repr(c) for c in df.columns] -- repr shows spaces"),
    ("ValueError (shape)", e_shape, "print .shape of BOTH operands; one of them was filtered"),
    ("TypeError (dtype)", e_dtype, "print df.dtypes; an object column is not a number"),
    ("TypeError (NaN)", e_nan_is_a_float, "df.isna().sum() -- missing text arrives as float NaN"),
    ("IndexError", e_indexerror, "print len() of the thing and the index you used, side by side"),
]

collected = []
for label, fn, first_check in cases:
    try:
        fn()
        last_line = "(no exception!)"
    except BaseException as exc:                      # noqa: BLE001 - teaching harness
        last_line = f"{type(exc).__name__}: {exc}".strip()
    collected.append({"what you did": label,
                      "the last line of the real traceback": last_line[:78],
                      "first thing to check": first_check})

table = pd.DataFrame(collected)
with pd.option_context("display.max_colwidth", 80, "display.width", 240):
    print(table.to_string(index=False))

         what you did                                            the last line of the real traceback                                          first thing to check
   KeyError (missing)                                                      KeyError: 'Ticket Number'     print list(df.columns) and diff it against what you typed
KeyError (whitespace)                                                           KeyError: 'Embarked'      print [repr(c) for c in df.columns] -- repr shows spaces
   ValueError (shape) ValueError: operands could not be broadcast together with shapes (714,) (891,)       print .shape of BOTH operands; one of them was filtered
    TypeError (dtype)                       TypeError: can only concatenate str (not "float") to str             print df.dtypes; an object column is not a number
      TypeError (NaN)                                 TypeError: 'float' object is not subscriptable          df.isna().sum() -- missing text arrives as float NaN
           IndexError 

Two of these deserve a second look because they are the ones that waste whole afternoons.

**`KeyError (whitespace)`.** The message is `'Embarked'` — exactly the name you typed, with nothing
in it to suggest a problem, because the stray space is in the *column name*, not in your string. The
column list looks correct when printed normally, which is what makes this one take an hour. `[repr(c) for c in df.columns]` is the two-second
habit that ends this class of bug forever, because `repr` shows you `'Embarked '` with the space
inside the quotes.

**`TypeError (NaN)`.** `'float' object is not subscriptable` while iterating over a column of
*strings* is the single most confusing message in the list, and the explanation is that pandas stores
missing text as `float('nan')`. There is no string there to subscript. Whenever a message mentions
`float` and your data is text, the word "float" means "missing".

In [11]:
# WHAT: demonstrate the two habits the table recommends, on the real columns.
# WHY: a recommendation you have not executed is a recommendation you will not remember.
print("repr() of the column names in the accidentally-renamed frame (last four):")
print("   ", [repr(c) for c in whitespace_df.columns][-4:])
print("   plain print() of the same names hides the problem completely:")
print("   ", list(whitespace_df.columns)[-4:])
print()
positions = [int(i) for i in np.flatnonzero(df["Embarked"].isna())]
print(f"rows whose Embarked is missing : {positions}  ({len(positions)} of {len(df)})")
print(f"type of the missing value      : {type(df['Embarked'].iloc[positions[0]]).__name__}")
print(f"type of a present value        : {type(df['Embarked'].iloc[0]).__name__}")
print()
print("That is the whole mystery: two of the 891 'strings' are floats, so port[0] fails on them.")

repr() of the column names in the accidentally-renamed frame (last four):
    ["'Ticket'", "'Fare'", "'Cabin'", "'Embarked '"]
   plain print() of the same names hides the problem completely:
    ['Ticket', 'Fare', 'Cabin', 'Embarked ']

rows whose Embarked is missing : [61, 829]  (2 of 891)
type of the missing value      : float
type of a present value        : str

That is the whole mystery: two of the 891 'strings' are floats, so port[0] fails on them.


## 5. The quiet four: failures that raise nothing at all

Every error above announced itself. **Those are the safe ones.** The dangerous failures in data work
produce a number, and the number is wrong, and nothing anywhere says so. There is no traceback to
read, which is precisely why a lesson on reading tracebacks has to cover them: you will otherwise
learn that "no exception" means "correct".

The four below are all produced live on the real manifest.

In [12]:
# WHAT: four real failures that produce a plausible WRONG answer instead of an exception.
# WHY: these are the ones that reach production, because nothing stops them.
df = pd.read_csv(TITANIC)

print("QUIET 1  -  SettingWithCopyWarning: the assignment that goes nowhere")
print("-" * 78)
third_class = df[df["Pclass"] == 3]                  # a filtered view, maybe a copy, maybe not
before = int(df["Age"].isna().sum())
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    third_class["Age"] = third_class["Age"].fillna(0.0)
    kinds = [w.category.__name__ for w in caught]
print(f"  warnings raised           : {kinds}")
print(f"  missing Age in the SLICE  : {int(third_class['Age'].isna().sum())}   (the fill worked here)")
print(f"  missing Age in the ORIGINAL, before {before} -> after {int(df['Age'].isna().sum())}"
      "   (the fill did NOT propagate)")
print("  It is a WARNING, not an error. The script continues. The cleaning silently did nothing")
print("  to the frame you go on to use. Correct forms: df.loc[mask, 'Age'] = ...  or  .copy() first.")

print()
print("QUIET 2  -  index misalignment: pandas aligns, and fills the gaps with NaN")
print("-" * 78)
adults = df[df["Age"] >= 18]
combined = df["Fare"] + adults["Age"]                # DIFFERENT indexes; pandas aligns them
print(f"  len(df)={len(df)}   len(adults)={len(adults)}   len(result)={len(combined)}")
print(f"  NaN in the result          : {int(combined.isna().sum())}")
print(f"  mean of the result         : {combined.mean():.4f}   <- computed from only "
      f"{int(combined.notna().sum())} rows")
print("  No error. No warning. A number came back and it is an average over a silently chosen subset.")

QUIET 1  -  SettingWithCopyWarning: the assignment that goes nowhere
------------------------------------------------------------------------------
  warnings raised           : ['SettingWithCopyWarning']
  missing Age in the SLICE  : 0   (the fill worked here)
  missing Age in the ORIGINAL, before 177 -> after 177   (the fill did NOT propagate)
  It is a WARNING, not an error. The script continues. The cleaning silently did nothing
  to the frame you go on to use. Correct forms: df.loc[mask, 'Age'] = ...  or  .copy() first.

QUIET 2  -  index misalignment: pandas aligns, and fills the gaps with NaN
------------------------------------------------------------------------------
  len(df)=891   len(adults)=601   len(result)=891
  NaN in the result          : 290
  mean of the result         : 68.9308   <- computed from only 601 rows
  No error. No warning. A number came back and it is an average over a silently chosen subset.


In [13]:
# WHAT: the other two quiet failures - a dtype that reads as text, and an off-by-one split.
# WHY: both give a confident, plausible, wrong answer on real data.
print("QUIET 3  -  dtype surprise: a numeric column parsed as text")
print("-" * 78)
as_text = pd.read_csv(TITANIC, dtype={"Fare": str})  # one bad quotechar in a real export does this
text_sum = as_text["Fare"].sum()
print(f"  dtype of Fare              : {as_text['Fare'].dtype}")
print(f"  as_text['Fare'].sum()      : a {type(text_sum).__name__} of length {len(text_sum)}, "
      f"starting {text_sum[:26]!r}")
print(f"  df['Fare'].sum() (correct) : {df['Fare'].sum():.2f}")
print("  + concatenated 891 strings instead of adding 891 numbers. Both are 'a sum'. One is money.")

print()
print("QUIET 4  -  off-by-one in a train/test split: the test set leaks into training")
print("-" * 78)
n = len(df)
cut = int(n * 0.8)
train = df.iloc[:cut + 1]                            # the +1 that nobody notices
test = df.iloc[cut:]
overlap = sorted(set(train.index) & set(test.index))
print(f"  n={n}  cut={cut}   len(train)={len(train)}  len(test)={len(test)}"
      f"   train+test={len(train) + len(test)}")
print(f"  rows in BOTH sets          : {len(overlap)}  -> index {overlap}")
print(f"  {len(overlap)} leaked row out of {len(test)} test rows = "
      f"{len(overlap) / len(test):.2%} of the test set is memorised, not predicted.")
print("  On a small test set that is enough to move a reported score. Nothing raises.")
print()
print("  The assertion that would have caught it, and costs one line:")
try:
    assert not set(train.index) & set(test.index), "train and test overlap"
except AssertionError as exc:
    print(f"    AssertionError: {exc}")

QUIET 3  -  dtype surprise: a numeric column parsed as text
------------------------------------------------------------------------------
  dtype of Fare              : object
  as_text['Fare'].sum()      : a str of length 4236, starting '7.2571.28337.92553.18.058.'
  df['Fare'].sum() (correct) : 28693.95
  + concatenated 891 strings instead of adding 891 numbers. Both are 'a sum'. One is money.

QUIET 4  -  off-by-one in a train/test split: the test set leaks into training
------------------------------------------------------------------------------
  n=891  cut=712   len(train)=713  len(test)=179   train+test=892
  rows in BOTH sets          : 1  -> index [712]
  1 leaked row out of 179 test rows = 0.56% of the test set is memorised, not predicted.
  On a small test set that is enough to move a reported score. Nothing raises.

  The assertion that would have caught it, and costs one line:
    AssertionError: train and test overlap


### What the quiet four have in common

None of them is a *Python* error. Every one is a violation of an assumption the code never wrote
down: that assigning to a slice changes the original; that two Series line up positionally; that a
column of digits is numeric; that `[:k]` and `[k:]` are disjoint. Python is not able to check
assumptions you have not stated.

Which gives the rule that turns this section into a habit:

> **Every time you make an assumption about shape, dtype, index, or disjointness, write it as an
> `assert` on the next line.** An `assert` costs one line and converts a silent wrong answer into a
> loud traceback — which, after this lesson, you can read in sixty seconds.

That is the whole trade: you cannot debug what does not fail, so make it fail.

## 6. `breakpoint()` and `pdb`

A traceback is a photograph of the moment of death. `pdb` lets you walk around inside the corpse and
ask it questions. There are two ways in.

**Ahead of time.** Put `breakpoint()` on the line before the trouble and run. Execution stops there
and you get a `(Pdb)` prompt. `breakpoint()` has been built in since Python 3.7, needs no import, and
obeys the `PYTHONBREAKPOINT` environment variable — so `PYTHONBREAKPOINT=0 python script.py` runs
straight past every breakpoint in the file without you editing anything. That is how you leave a
breakpoint in code you are still working on without risking it in a batch run.

**After the fact — a *post-mortem*.** The failure already happened and you do not want to run the
whole job again. `pdb.post_mortem()` reopens the stack of the exception that just escaped, with every
local variable still alive. In Jupyter the same thing is the `%debug` magic, typed into a fresh cell
immediately after a cell has failed.

Post-mortem is the one to reach for first, because it costs nothing: the crash has already paid for
it.

The next cell runs a **real post-mortem session**. Because a notebook being executed by a script has
nobody at the keyboard, the cell starts a fresh interpreter, lets it fail, and feeds `pdb` a prepared
list of commands on standard input. What prints is the genuine transcript — `(Pdb)` prompts and all —
exactly what you would see after typing the same eight commands in a terminal.

(It uses a subprocess for a second reason. Inside IPython the name `pdb.Pdb` has been replaced by a
coloured variant with an `ipdb>` prompt. That is a fine debugger, but it is not the one waiting for
you on a server, so this lesson shows you the plain one.)

In [14]:
# WHAT: run a genuine standard-library pdb post-mortem on the failure from Section 1 and print
# the real session transcript.
# WHY: you cannot type into a notebook that a script is executing, but the SESSION is the lesson.
# pdb_session() (defined in the setup cell) runs it in a fresh interpreter and feeds it commands.
commands = [
    "w",                          # where am I? print the stack, with > marking the current frame
    "p len(ages)",                # print an expression evaluated in the CURRENT frame
    "p len(df)",
    "p ages.index[3:9].tolist()", # the labels survived dropna -- they are NOT 0, 1, 2, 3, ...
    "u",                          # move UP one frame, into the caller
    "ll",                         # longlist: the whole of the current function
    "p df.shape",
    "q",                          # quit
]

transcript = pdb_session(
    f"import pipeline\npipeline.report({str(TITANIC)!r})",
    "\n".join(commands) + "\n",
)

print("=" * 78)
print("REAL pdb POST-MORTEM TRANSCRIPT")
print("commands typed: " + " / ".join(commands))
print("=" * 78)
print(transcript)

REAL pdb POST-MORTEM TRANSCRIPT
commands typed: w / p len(ages) / p len(df) / p ages.index[3:9].tolist() / u / ll / p df.shape / q
> lab/pipeline.py(17)age_times_fare()
-> return ages.to_numpy() * df["Fare"].to_numpy()
(Pdb)   lab/pdb_driver.py(5)<module>()
-> pipeline.report('/Users/abdullah/Downloads/AI Diploma/Course 04/datasets/raw/titanic.csv')
  lab/pipeline.py(24)report()
-> return float(age_times_fare(ages, df).mean())
> lab/pipeline.py(17)age_times_fare()
-> return ages.to_numpy() * df["Fare"].to_numpy()
(Pdb) 714
(Pdb) 891
(Pdb) [3, 4, 6, 7, 8, 9]
(Pdb) > lab/pipeline.py(24)report()
-> return float(age_times_fare(ages, df).mean())
(Pdb)  20  	def report(csv_path):
 21  	    """The whole job: load, clean, combine, average."""
 22  	    df = load_passengers(csv_path)
 23  	    ages = clean_ages(df)
 24  ->	    return float(age_times_fare(ages, df).mean())
(Pdb) (891, 12)
(Pdb)


Read the transcript above and notice what it bought. `w` printed the stack and marked the current
frame with `>`. `p len(ages)` and `p len(df)` answered, from live memory, the exact question the
`ValueError` posed — the two numbers in the exception message, but now attached to **names**, so you
know *which* variable is the short one. `p ages.index[3:9].tolist()` printed the labels of six
consecutive elements and they are not consecutive numbers: one is missing, because that passenger's
age was dropped. That is the defect, visible directly, in one command. Then `u` walked up into
`report`, where `ll` listed the whole function and `p df.shape` confirmed the frame above still had
all 891 rows.

That is the entire diagnosis of Section 2 — seven commands and a quit, no edit to any file, and no
re-running of the job.

### The commands worth memorising

| command | short | what it does |
|---|---|---|
| `where` | `w` | print the stack; `>` marks where you are |
| `up` / `down` | `u` / `d` | move one frame towards the caller / towards the crash |
| `list` / `longlist` | `l` / `ll` | source around the current line / the whole current function |
| `print` / `pp` | `p` / `pp` | evaluate an expression here; `pp` pretty-prints |
| `args` | `a` | the arguments this frame was called with |
| `next` | `n` | run the next line, stepping *over* calls |
| `step` | `s` | run the next line, stepping *into* calls |
| `continue` | `c` | run until the next breakpoint or the end |
| `interact` | — | drop into a full Python REPL with this frame's locals |
| `quit` | `q` | leave |

Two traps worth knowing before you meet them. A variable named `n`, `c`, `s`, `l`, `a` or `q` cannot
be inspected by typing its name — `pdb` reads it as the command — so use `p n`. And `p` **evaluates**
what you type, so `p df.drop(columns="Age")` really will run; keep post-mortem expressions read-only.

### The same traceback outside Jupyter

Notebooks are a comfortable place to see a traceback and a misleading one, because the notebook
catches it, formats it in colour, and keeps the kernel alive. In a terminal — where the code you are
handed will actually run — the process **dies**, the traceback goes to **stderr**, and the shell gets
a **non-zero exit status**. That exit status is the thing CI, `make`, and every scheduler in the
world are watching. The next cell runs a real failing script in a real shell to show all three.

In [15]:
%%bash
# WHAT: run a genuinely failing one-liner in a real shell and inspect what the shell sees.
# WHY: outside Jupyter a traceback is stderr plus an exit code, and the exit code is what automation
# reads. $LAB_PY and $LAB_DIR were exported by the setup cell.
cd "$LAB_DIR"

cat > tiny_fail.py <<'EOF'
import pandas as pd
import sys

df = pd.read_csv(sys.argv[1])
print("rows:", len(df))
print("mean fare per head:", df["Fare"].mean() / df["Party Size"].mean())
EOF

echo "--- stdout and stderr together, as you would normally see them ---"
"$LAB_PY" tiny_fail.py "$TITANIC_PATH" 2>&1 | tail -n 8

echo
echo "--- stdout ONLY (stderr discarded): the traceback vanishes, the wrong impression remains ---"
"$LAB_PY" tiny_fail.py "$TITANIC_PATH" 2>/dev/null

echo
"$LAB_PY" tiny_fail.py "$TITANIC_PATH" >/dev/null 2>&1
echo "--- exit status of the failing run: $?   (0 means success; anything else is a failure) ---"

--- stdout and stderr together, as you would normally see them ---


  File "/private/var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/traceback_lab_b40amygd/tiny_fa

il.py", line 6, in <module>
    print("mean fare per head:", df["Fare"].mean()

 / df["Party Size"].mean())
                                                    

 ~~^^^^^^^^^^^^^^
  File "/Users/abdullah/Downloads/AI Diploma/.venv/lib/py

...

3.14/site-packages/pandas/core/indexes/base.py", line 3819, in get_loc
    rai

se KeyError(key) from err
KeyError: 'Party Size'



--- stdout ONLY (stderr discarded): the traceback vanishes, the wrong impression remains ---


rows: 891


--- exit status of the failing run: 1   (0 means success; anything else is a failure) ---


Three things to take from that shell run.

The traceback went to **stderr**, so `2>/dev/null` made it disappear while `rows: 891` still printed
happily. A pipeline that logs only stdout will show you a job that looked like it was working right
up until it stopped producing files.

The exit status was **1**. Every failing Python process exits non-zero. `echo $?` immediately after a
command is the fastest question you can ask a shell, and `cmd && next_step` will refuse to run
`next_step` after a failure for exactly this reason.

And the traceback itself has exactly the **shape** you learned to read in Section 1 — the same
banner, the same frames oldest-first, the same PEP 657 carets, the same `Type: message` last line —
even though this is a different failure (`KeyError: 'Party Size'`) in a different program. The
structure is a property of Python, not of Jupyter, so the skill transfers unchanged. That is the
point of practising it here.

## 7. Bisecting a failure

When the traceback names a line but you still cannot see why, stop reading and start **halving**. A
failure that depends on the input is a search problem, and search problems have a best algorithm.

Bisection needs one property and one property only: the answer must be **monotone** — true, true,
true, then false, false, false, with a single flip. Then each probe halves the space: 891 candidates
cost **10 probes**, whatever the answer turns out to be. A linear scan costs however many rows happen
to precede the culprit — which is luck, and in the worst case is all 891.

### 7a. Bisect the data

The question below is a real one about the real manifest: *processing the passengers in file order,
which is the first row that breaks this function?* The predicate "the first `k` rows all survive" is
monotone, so bisection applies.

In [16]:
# WHAT: find the first offending row of 891 by halving, and count the probes honestly.
# WHY: bisection is the difference between a two-minute diagnosis and an afternoon of print().
df = pd.read_csv(TITANIC)


def first_letter_of_every_port(rows):
    """The function under suspicion. Fine for most passengers; not for all of them."""
    return [port[0] for port in rows["Embarked"]]


def survives_first_k(k):
    """Monotone predicate: do the first k rows all get through?"""
    try:
        first_letter_of_every_port(df.iloc[:k])
        return True
    except Exception:
        return False


# --- the linear way, for comparison only -------------------------------------
linear_probes = 0
for k in range(len(df) + 1):
    linear_probes += 1
    if not survives_first_k(k):
        linear_answer = k - 1
        break

# --- bisection ---------------------------------------------------------------
lo, hi = 0, len(df)          # survives_first_k(0) is True; survives_first_k(891) is False
bisect_probes = 0
trace = []
while hi - lo > 1:
    mid = (lo + hi) // 2
    bisect_probes += 1
    ok = survives_first_k(mid)
    trace.append((bisect_probes, lo, mid, hi, "survives" if ok else "breaks"))
    if ok:
        lo = mid
    else:
        hi = mid
culprit = lo                 # row index lo is the last good one -> row lo is the first bad row

print(f"{'probe':>5} {'lo':>5} {'mid':>5} {'hi':>5}   result")
for p, l, m, h, res in trace:
    print(f"{p:>5} {l:>5} {m:>5} {h:>5}   first {m} rows {res}")

print()
print(f"bisection: {bisect_probes} probes over {len(df)} rows  ->  first failing row index = {culprit}")
print(f"linear scan: {linear_probes} probes for the same answer ({linear_answer}) -- and it was")
print(f"             lucky: had the culprit been the last row it would have cost {len(df) + 1}.")
print(f"bisection's cost does not depend on luck: ceil(log2({len(df)})) = "
      f"{int(np.ceil(np.log2(len(df))))} probes, always.")
print()
bad_row = df.iloc[culprit]
print(f"row {culprit}: {bad_row['Name']}")
print(f"   Embarked = {bad_row['Embarked']!r}   type = {type(bad_row['Embarked']).__name__}")
_ = show_traceback(first_letter_of_every_port, df.iloc[culprit:culprit + 1])

probe    lo   mid    hi   result
    1     0   445   891   first 445 rows breaks
    2     0   222   445   first 222 rows breaks
    3     0   111   222   first 111 rows breaks
    4     0    55   111   first 55 rows survives
    5    55    83   111   first 83 rows breaks
    6    55    69    83   first 69 rows breaks
    7    55    62    69   first 62 rows breaks
    8    55    58    62   first 58 rows survives
    9    58    60    62   first 60 rows survives
   10    60    61    62   first 61 rows survives

bisection: 10 probes over 891 rows  ->  first failing row index = 61
linear scan: 63 probes for the same answer (61) -- and it was
             lucky: had the culprit been the last row it would have cost 892.
bisection's cost does not depend on luck: ceil(log2(891)) = 10 probes, always.

row 61: Icard, Miss. Amelie
   Embarked = nan   type = float
Traceback (most recent call last):
  File "<this notebook>", line 8, in first_letter_of_every_port
    return [port[0] for port in rows

Read the probe table as a decision log. Each row halves the interval, the interval never widens, and
the process cannot fail to terminate. That is why bisection is worth reaching for even when you
*think* you know the answer: it is one of the very few debugging techniques with a guaranteed bound
on how long it takes, and that bound does not depend on how lucky you are.

Then look at what the last two lines did. Having found the row, the notebook re-ran the failing
function on **that single row** and printed its traceback. A one-row reproduction is the ideal end
state of any bug hunt: it is fast, it is deterministic, it fits in a bug report, and it becomes a
regression test with almost no extra work.

### 7b. Bisect the history: `git bisect run`

The same algorithm applies to time. *It worked last month and it is broken now, and there are a
hundred commits in between.* Git has bisection built in, and it can drive it automatically:

```bash
git bisect start
git bisect bad                 # the current commit is broken
git bisect good v1.2.0         # this old commit/tag was fine
git bisect run pytest -x tests/test_thing.py     # git checks out commits and runs this for you
git bisect reset               # ALWAYS: puts you back where you started
```

The contract of the script you pass to `git bisect run` is only about its **exit status**, and the
documentation is exact about it: exit `0` if this commit is good/old; exit with a code between `1`
and `127` inclusive, **except 125**, if it is bad/new; use `125` when this commit cannot be tested,
which skips it. Any other exit code aborts the bisection. Nothing else matters — which is exactly why
Section 6's point about exit codes was worth making, and why any failing Python script works as a
bisect test with no adaptation at all.

The cell below builds a **real 12-commit git repository in the scratch directory**, hides a
one-character regression in one of the commits, and lets `git bisect run` find it. Nothing touches
this repository: `GIT_CONFIG_GLOBAL` is pointed at a non-existent file so your own git settings are
neither read nor written, and the whole thing is deleted at the end.

In [17]:
# WHAT: create a throwaway git repo with a real regression buried in its history, then let
# `git bisect run` find the guilty commit automatically.
# WHY: this is the highest-leverage git command most people never learn, and it is one command.
import subprocess

SANDBOX = LAB / "history_sandbox"
SANDBOX.mkdir(exist_ok=True)

# Isolate completely from the machine's git configuration and from any credential prompt.
GENV = {**os.environ,
        "GIT_AUTHOR_NAME": "lab", "GIT_AUTHOR_EMAIL": "lab@example.invalid",
        "GIT_COMMITTER_NAME": "lab", "GIT_COMMITTER_EMAIL": "lab@example.invalid",
        "GIT_CONFIG_GLOBAL": str(SANDBOX / "no-such-gitconfig"),
        "GIT_CONFIG_SYSTEM": str(SANDBOX / "no-such-gitconfig"),
        "GIT_TERMINAL_PROMPT": "0"}


def git(*args, check=True):
    """Run one git command inside the sandbox and hand back the completed process."""
    done = subprocess.run(["git", "-C", str(SANDBOX), *args],
                          capture_output=True, text=True, env=GENV)
    if check and done.returncode != 0:
        raise RuntimeError(" ".join(args) + "\n" + done.stderr)
    return done


subprocess.run(["git", "init", "-q", "-b", "main", str(SANDBOX)],
               capture_output=True, text=True, env=GENV, check=True)

WORKING = 'def survival_rate(survived, total):\n    """Share of passengers who survived."""\n    return survived / total\n'
BROKEN = 'def survival_rate(survived, total):\n    """Share of passengers who survived."""\n    return survived // total\n'   # / became //

# The test the bisect will run: real numbers from the real manifest. 342 of 891 survived.
survivors, passengers = int(df["Survived"].sum()), len(df)
TEST = textwrap.dedent(f'''
    """Exit 0 if survival_rate is right, 1 if it is wrong. That is the whole contract."""
    import sys
    sys.path.insert(0, ".")
    from metrics import survival_rate
    expected = {survivors} / {passengers}
    sys.exit(0 if abs(survival_rate({survivors}, {passengers}) - expected) < 1e-12 else 1)
''').lstrip()
(SANDBOX / "test_survival_rate.py").write_text(TEST, encoding="utf-8")

REGRESSION_AT = 7
for i in range(12):
    (SANDBOX / "metrics.py").write_text(BROKEN if i >= REGRESSION_AT else WORKING, encoding="utf-8")
    (SANDBOX / "notes.md").write_text(f"routine edit number {i}\n", encoding="utf-8")
    git("add", "-A")
    git("-c", "commit.gpgsign=false", "commit", "-q", "-m", f"commit {i:02d}: routine edit")

history = git("log", "--oneline", "--reverse").stdout.strip().splitlines()
oldest, newest = history[0].split()[0], history[-1].split()[0]
print(f"built a sandbox repository with {len(history)} commits; the regression is in exactly one of them")
print(f"the test script decides good/bad purely by exit code, using the real figures "
      f"{survivors}/{passengers} survivors")
print()

git("bisect", "start")
git("bisect", "bad", newest)
git("bisect", "good", oldest)
run = subprocess.run(["git", "-C", str(SANDBOX), "bisect", "run", sys.executable, "test_survival_rate.py"],
                     capture_output=True, text=True, env=GENV)
git("bisect", "reset", check=False)

print("=" * 78)
print("REAL `git bisect run` OUTPUT")
print("=" * 78)
print(run.stdout.strip())

tested = sum(1 for line in run.stdout.splitlines() if line.startswith("running"))
print()
print(f"commits git actually tested : {tested}  (out of {len(history) - 1} candidates)")
print(f"a linear walk would have needed up to {len(history) - 1} test runs; "
      f"log2({len(history) - 1}) = {np.log2(len(history) - 1):.1f}")

built a sandbox repository with 12 commits; the regression is in exactly one of them
the test script decides good/bad purely by exit code, using the real figures 342/891 survivors

REAL `git bisect run` OUTPUT
running '/Users/abdullah/Downloads/AI Diploma/.venv/bin/python' 'test_survival_rate.py'
Bisecting: 2 revisions left to test after this (roughly 2 steps)
[0bcfc12fdcb51bb0adb9edaf37774a1a7054f476] commit 08: routine edit
running '/Users/abdullah/Downloads/AI Diploma/.venv/bin/python' 'test_survival_rate.py'
Bisecting: 0 revisions left to test after this (roughly 1 step)
[d5e32f695d92ac50aeea9f47bedc855c3617ee81] commit 07: routine edit
running '/Users/abdullah/Downloads/AI Diploma/.venv/bin/python' 'test_survival_rate.py'
Bisecting: 0 revisions left to test after this (roughly 0 steps)
[388a2f85aeddebd1222c8fa60d748a47379b2970] commit 06: routine edit
running '/Users/abdullah/Downloads/AI Diploma/.venv/bin/python' 'test_survival_rate.py'
d5e32f695d92ac50aeea9f47bedc855c3617ee81 is

Read the output above from the bottom: `... is the first bad commit`, followed by the commit's full
message and — the part that matters — its **diff stat**, naming the files it touched. `git bisect`
does not tell you what is wrong. It tells you *which change to read*. Here it narrowed eleven
candidate commits to one; the same command narrows a thousand commits to one in about ten test runs,
because that is what halving does. Reading one diff is a job. Reading a thousand is not.

Two practical notes. **Always finish with `git bisect reset`**, or you are left on a detached HEAD in
the middle of history, which is how people lose an afternoon and think git ate their work. And a
bisect is only as good as its test: if your test is flaky, bisect will confidently name an innocent
commit, because the algorithm assumes the answer flips exactly once.

## 8. The anchor: this repository's own lying traceback

Now reproduce the bug from the opening section, on the real file, with the real code.

`tools/verify/parse_baseline.py` needs to syntax-check notebook cells that may contain IPython
magics. Its first version took the obvious approach — replace any line starting with `%` or `!` with
`pass`, then compile — and that approach has a hole in it. The `%` character is also Python's
oldest string-formatting operator, and in a multi-line expression a continuation line may perfectly
legitimately *start* with it.

The cell below defines both versions verbatim, loads the real cell 10 of the real notebook, and runs
both.

In [18]:
# WHAT: reproduce a real false-positive from this repository's own tooling, on the real notebook cell.
# WHY: it is the clearest example in the repo of a traceback whose every fact is true and whose
# obvious conclusion is wrong.
import json

VICTIM = REPO / "Course 08" / "unit2-cnns" / "examples" / "07_training_cnn_image_datasets.ipynb"
notebook = json.loads(VICTIM.read_text(encoding="utf-8"))
cell_source = notebook["cells"][10]["source"]
CELL = "".join(cell_source) if isinstance(cell_source, list) else cell_source


def strip_magics_v1(src):
    """The original, 2026-08. Any line starting with % or ! is not Python -- or so it assumed."""
    lines = src.splitlines()
    if lines and lines[0].lstrip().startswith("%%"):
        return ""
    return "\n".join("pass" if ln.lstrip().startswith(("%", "!")) else ln for ln in lines)


def strip_magics_v2(src):
    """The fix: track bracket depth. Inside an open bracket, % is the format operator."""
    lines = src.splitlines()
    if lines and lines[0].lstrip().startswith("%%"):
        return ""
    out, depth = [], 0
    for ln in lines:
        out.append("pass" if (depth == 0 and ln.lstrip().startswith(("%", "!"))) else ln)
        depth += ln.count("(") + ln.count("[") + ln.count("{")
        depth -= ln.count(")") + ln.count("]") + ln.count("}")
        depth = max(depth, 0)
    return "\n".join(out)


print(f"victim: {VICTIM.relative_to(REPO)}")
print(f"cell 10 is {len(CELL.splitlines())} lines of real, working, executed notebook code")
print()
for name, strip in [("v1  (the original)", strip_magics_v1), ("v2  (bracket-depth fix)", strip_magics_v2)]:
    candidate = strip(CELL)
    try:
        compile(candidate, "cell10", "exec")
        verdict = "compiles OK"
    except SyntaxError as err:
        verdict = f"SYNTAX cell 10: {err.msg} (line {err.lineno})"
    print(f"{name:<24} -> {verdict}")

changed = [i for i, (real, seen) in enumerate(zip(CELL.splitlines(), strip_magics_v1(CELL).splitlines()), 1)
           if real != seen]
print()
print(f"lines v1 rewrote to 'pass': {changed}")
print()
print("The real cell, lines 20-23 -- the ONLY lines involved:")
for n, ln in enumerate(CELL.splitlines()[19:23], start=20):
    flag = "  <-- v1 replaced this whole line with 'pass'" if n in changed else ""
    flag = flag or ("  <-- and the error was reported HERE" if n == 23 else "")
    print(f"{n:>3} | {ln}{flag}")

victim: Course 08/unit2-cnns/examples/07_training_cnn_image_datasets.ipynb
cell 10 is 23 lines of real, working, executed notebook code

v1  (the original)       -> SYNTAX cell 10: unmatched ')' (line 23)
v2  (bracket-depth fix)  -> compiles OK

lines v1 rewrote to 'pass': [22]

The real cell, lines 20-23 -- the ONLY lines involved:
 20 |     plt.tight_layout(); plt.show()
 21 |     print("last epoch: train loss %.4f / val loss %.4f | train acc %.4f / val acc %.4f"
 22 |           % (history.history["loss"][-1], history.history["val_loss"][-1],  <-- v1 replaced this whole line with 'pass'
 23 |              history.history["accuracy"][-1], history.history["val_accuracy"][-1]))  <-- and the error was reported HERE


In [19]:
# WHAT: sweep every course notebook in the repository with both versions and count the disagreement.
# WHY: one anecdote is a story; a count over the whole corpus is evidence about how big the hole was.
import time

start = time.time()
scanned, only_v1_fails, both_fail = 0, [], []
for path in sorted(REPO.glob("Course */**/*.ipynb")):
    if ".ipynb_checkpoints" in path.parts:
        continue
    try:
        book = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue
    scanned += 1
    for index, cell in enumerate(book.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue
        src = cell.get("source")
        src = "".join(src) if isinstance(src, list) else (src or "")
        if not src.strip():
            continue

        def compiles(strip):
            text = strip(src)
            if not text.strip():
                return True
            try:
                compile(text, "x", "exec")
                return True
            except SyntaxError:
                return False

        ok1, ok2 = compiles(strip_magics_v1), compiles(strip_magics_v2)
        if ok2 and not ok1:
            only_v1_fails.append((path.relative_to(REPO), index))
        elif not ok1 and not ok2:
            both_fail.append((path.relative_to(REPO), index))

print(f"scanned {scanned} course notebooks in {time.time() - start:.1f} s")
print(f"cells v1 rejected that v2 accepts (FALSE alarms) : {len(only_v1_fails)}"
      f"  across {len({p for p, _ in only_v1_fails})} notebooks")
print(f"cells BOTH versions reject                       : {len(both_fail)}")
print()
print("the notebooks v1 falsely accused:")
for path, index in only_v1_fails:
    print(f"   {path}  cell {index}")

scanned 421 course notebooks in 1.0 s
cells v1 rejected that v2 accepts (FALSE alarms) : 12  across 8 notebooks
cells BOTH versions reject                       : 0

the notebooks v1 falsely accused:
   Course 06/unit1-ethics-foundations/examples/03_case_study_analysis.ipynb  cell 8
   Course 06/unit1-ethics-foundations/examples/03_case_study_analysis.ipynb  cell 11
   Course 08/unit2-cnns/examples/01_cnn_architecture.ipynb  cell 10
   Course 08/unit2-cnns/examples/03_cnn_advanced_architectures.ipynb  cell 11
   Course 08/unit2-cnns/examples/04_transfer_learning_object_detection.ipynb  cell 12
   Course 08/unit2-cnns/examples/05_transfer_learning_cnns.ipynb  cell 12
   Course 08/unit2-cnns/examples/07_training_cnn_image_datasets.ipynb  cell 10
   Course 08/unit2-cnns/examples/07_training_cnn_image_datasets.ipynb  cell 15
   Course 08/unit2-cnns/examples/07_training_cnn_image_datasets.ipynb  cell 19
   Course 10/unit4-ethics-regulations/examples/04_applying_ai_regulatory_guidelines_gdpr

### The post-mortem, at three levels

**Where it broke.** `compile()`, at line 23 of the cell. True, and useless.

**Where it went wrong.** Line 22 of the cell was replaced by `pass`, so the `print(` opened on line
21 was never closed and line 23's closing brackets had nothing to match. Line 23 was simply the first
place the mutilated text stopped being valid Python. That is what a syntax error *always* reports:
not where the mistake is, but where the parser first noticed.

**Where the bug actually was.** In `strip_magics`, in a different file, which nothing in the
traceback mentioned at all. The fix is the bracket-depth counter in `v2`, and the sweep above
measures what it recovered.

### The general rule, which is worth more than the specific fix

> **When a traceback describes code you cannot see a problem in, check that the string being executed
> is the string you think it is.**

The interpreter is never wrong about the text it was given. It can only be wrong about which text
that was — and the layers that can substitute one for another are everywhere: a preprocessor, a
template engine, a code generator, `exec()` on a built-up string, a notebook's own magic
transformation, a stale `.pyc`, a shadowed module earlier on `sys.path`, an editor that saved
somewhere else. In every one of these cases the traceback is honest and the reader is misled.

The two-second check that resolves it, for a module: print `module.__file__` and open **that** file.
For generated or transformed source: print the string you are about to compile, with line numbers,
and read line 23 yourself. Printing the text that `compile()` was actually handed is what ends
Section 8's investigation; without it, no amount of staring at the notebook will.

Notice, too, that `strip_magics` had no test. A single test asserting that a cell containing a
multi-line `%`-format expression still compiles would have caught this on the day it was written, and
would have cost fewer lines than this paragraph.

## 🛠️ Your turn

Five tasks. Each one has a checker underneath that verifies your work **without showing you the
answer** — the expected values are stored as hashes, so guessing is not open to you and copying is
not available. Edit only the lines marked `# YOUR CODE`.

Run them as they are first. They will all report `not yet`, which is the correct starting state.

In [20]:
# WHAT: the shared checker used by all five tasks.
# WHY: it must confirm a right answer without ever displaying one.
import hashlib


def fingerprint(value):
    """Stable short hash of an answer, so the notebook can check without revealing."""
    return hashlib.sha256(repr(value).encode()).hexdigest()[:12]


def check(task, got, expected_fingerprint, hint):
    ok = fingerprint(got) == expected_fingerprint
    print(f"{'PASS' if ok else 'not yet'}  {task}")
    print(f"      you produced: {got!r}")
    if not ok:
        print(f"      hint: {hint}")
    return ok


SCORE = {}
print("checker ready - each task below reports on its own")

checker ready - each task below reports on its own


In [21]:
# ---------------------------------------------------------------------------
# TASK 1 - read a traceback you have not seen before.
# Run the cell. A real traceback prints. Answer the three questions by editing
# the three values, then run it again.
# ---------------------------------------------------------------------------
def fare_per_family_member(frame):
    party = frame["SibSp"] + frame["Parch"] + 1
    cabins = frame["Cabin"].dropna()
    return (frame["Fare"].to_numpy() / party.to_numpy()) + cabins.str.len().to_numpy()


print("Read this traceback, then answer below.")
print("-" * 78)
show_traceback(fare_per_family_member, pd.read_csv(TITANIC))
print("-" * 78)

# Q1: what is the exception TYPE?  (a string, e.g. "KeyError")
T1_TYPE = "???"                                   # YOUR CODE

# Q2: in which FUNCTION did execution stop? (the name in the last frame, no brackets)
T1_FUNCTION = "???"                               # YOUR CODE

# Q3: the message names two lengths. Which is the SHORTER one? (an int)
T1_SHORTER = 0                                    # YOUR CODE

SCORE["task 1"] = all([
    check("task 1a - exception type", T1_TYPE, "3200aae08de1", "the last line, left of the colon"),
    check("task 1b - crash function", T1_FUNCTION, "26d2a7cf44cf",
          "the LAST 'in <name>' in the frame list, not the first"),
    check("task 1c - shorter length", T1_SHORTER, "fc56dbc6d465",
          "count the non-null Cabin values with df['Cabin'].notna().sum()"),
])

Read this traceback, then answer below.
------------------------------------------------------------------------------
Traceback (most recent call last):
  File "<this notebook>", line 9, in fare_per_family_member
    return (frame["Fare"].to_numpy() / party.to_numpy()) + cabins.str.len().to_numpy()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~~~~~~~~~~
ValueError: operands could not be broadcast together with shapes (891,) (204,)
------------------------------------------------------------------------------
not yet  task 1a - exception type
      you produced: '???'
      hint: the last line, left of the colon
not yet  task 1b - crash function
      you produced: '???'
      hint: the LAST 'in <name>' in the frame list, not the first
not yet  task 1c - shorter length
      you produced: 0
      hint: count the non-null Cabin values with df['Cabin'].notna().sum()


In [22]:
# ---------------------------------------------------------------------------
# TASK 2 - repair the off-by-one split from Section 5 so that the two sets are
# disjoint AND together cover all 891 rows. Change only the two slice lines.
# ---------------------------------------------------------------------------
df = pd.read_csv(TITANIC)
cut = int(len(df) * 0.8)

train_set = df.iloc[:cut + 1]                     # YOUR CODE
test_set = df.iloc[cut:]                          # YOUR CODE

overlap = len(set(train_set.index) & set(test_set.index))
covered = len(set(train_set.index) | set(test_set.index))
print(f"train={len(train_set)}  test={len(test_set)}  overlap={overlap}  rows covered={covered}")

SCORE["task 2"] = check("task 2 - a clean split", (overlap, covered), "2a98224d7acc",
                        "you want (overlap, covered) == (0, 891); check both slice bounds")

train=713  test=179  overlap=1  rows covered=891
not yet  task 2 - a clean split
      you produced: (1, 891)
      hint: you want (overlap, covered) == (0, 891); check both slice bounds


In [23]:
# ---------------------------------------------------------------------------
# TASK 3 - bisect the data. `unit_fare` fails on some passengers. Find the FIRST
# failing row index by BISECTION, using at most 12 probes. A linear scan will be
# rejected by the probe budget.
# ---------------------------------------------------------------------------
df = pd.read_csv(TITANIC)
PROBES = 0


def unit_fare(rows):
    """Cost per person. Fails on the passengers who travelled on a zero fare."""
    return [1.0 / float(fare) for fare in rows["Fare"]]


def survives_first_k(k):
    """Monotone predicate. Every call is counted against your probe budget."""
    global PROBES
    PROBES += 1
    try:
        unit_fare(df.iloc[:k])
        return True
    except Exception:
        return False


lo, hi = 0, len(df)
# YOUR CODE: halve [lo, hi) until hi - lo == 1, using survives_first_k(mid)

first_bad_row = lo

print(f"probes used: {PROBES}  (budget 12)   answer: {first_bad_row}")
found = check("task 3 - first failing row", first_bad_row, "3068430da9e4",
              "while hi - lo > 1: mid = (lo + hi) // 2; then move lo or hi to mid")
within_budget = PROBES <= 12
if not within_budget:
    print(f"      rejected: {PROBES} probes is over budget - that is a scan, not a bisection")
SCORE["task 3"] = found and within_budget

probes used: 0  (budget 12)   answer: 0
not yet  task 3 - first failing row
      you produced: 0
      hint: while hi - lo > 1: mid = (lo + hi) // 2; then move lo or hi to mid


In [24]:
# ---------------------------------------------------------------------------
# TASK 4 - post-mortem with pdb. The function below fails. Use a real pdb session
# to find how many DISTINCT deck letters the local variable `deck` holds at the
# moment of the crash, and put that number in T4_DISTINCT_DECKS.
# Recover it from the transcript, not by computing it outside the debugger.
# ---------------------------------------------------------------------------
BROKEN_CASE = f"""
import pandas as pd


def cabin_decks(frame):
    deck = frame["Cabin"].dropna().str[0]
    return deck.to_numpy() + frame["Pclass"].to_numpy()      # str + int, element-wise


cabin_decks(pd.read_csv({str(TITANIC)!r}))
"""

task4_commands = [
    # YOUR CODE: add pdb commands here, BEFORE the "q"
    "q",
]

print(pdb_session(BROKEN_CASE, "\n".join(task4_commands) + "\n"))
print("-" * 78)

T4_DISTINCT_DECKS = 0                             # YOUR CODE: the number your pdb session printed

SCORE["task 4"] = check("task 4 - distinct decks at the crash", T4_DISTINCT_DECKS, "2c624232cdd2",
                        "'p deck.nunique()' before the 'q'; then re-run and read the transcript")

> lab/pdb_driver.py(10)cabin_decks()
-> return deck.to_numpy() + frame["Pclass"].to_numpy()      # str + int, element-wise
(Pdb)
------------------------------------------------------------------------------
not yet  task 4 - distinct decks at the crash
      you produced: 0
      hint: 'p deck.nunique()' before the 'q'; then re-run and read the transcript


In [25]:
# ---------------------------------------------------------------------------
# TASK 5 - a silent failure. The code below raises nothing and returns a number.
# The number is wrong. Report how many of the 891 products are NaN because of the
# index misalignment, and write the ONE-LINE assert that would have caught it.
# ---------------------------------------------------------------------------
df = pd.read_csv(TITANIC)
first_class = df[df["Pclass"] == 1]
suspicious = df["Fare"] * first_class["Age"]      # two different indexes; pandas aligns silently

print(f"no exception. mean of the result = {suspicious.mean():.4f}  over "
      f"{int(suspicious.notna().sum())} usable values")

T5_NAN_COUNT = 0                                  # YOUR CODE: how many of the 891 products are NaN?

# How many of df's index labels are absent from first_class? Those rows can never
# produce a product, whatever their Age says.
T5_LABELS_NOT_IN_FIRST_CLASS = 0                  # YOUR CODE

SCORE["task 5"] = all([
    check("task 5a - NaN count", T5_NAN_COUNT, "bd94717d9126",
          "suspicious.isna().sum() - predict it before you measure it"),
    check("task 5b - unmatched index labels", T5_LABELS_NOT_IN_FIRST_CLASS, "a440868cf431",
          "len(set(df.index) - set(first_class.index))"),
])
print()
print("When both pass, check the arithmetic: 5b + (first-class passengers with no Age) should")
print(f"equal 5a exactly. first-class passengers with no recorded Age = {int(first_class['Age'].isna().sum())}")

no exception. mean of the result = 3102.8217  over 186 usable values
not yet  task 5a - NaN count
      you produced: 0
      hint: suspicious.isna().sum() - predict it before you measure it
not yet  task 5b - unmatched index labels
      you produced: 0
      hint: len(set(df.index) - set(first_class.index))

When both pass, check the arithmetic: 5b + (first-class passengers with no Age) should
equal 5a exactly. first-class passengers with no recorded Age = 30


In [26]:
# WHAT: scoreboard, then delete the scratch directory.
# WHY: a lab that does not clean up after itself is a lab that pollutes the next person's machine.
print("=" * 78)
for name, passed in SCORE.items():
    print(f"  {'PASS' if passed else 'not yet'}   {name}")
print(f"  {sum(SCORE.values())} of {len(SCORE)} complete")
print("=" * 78)
print()

sys.modules.pop("pipeline", None)
if str(LAB) in sys.path:
    sys.path.remove(str(LAB))
shutil.rmtree(LAB, ignore_errors=True)
print(f"scratch directory removed: {not LAB.exists()}")
print(f"repository untouched: nothing outside {LAB.name} was written at any point")

  not yet   task 1
  not yet   task 2
  not yet   task 3
  not yet   task 4
  not yet   task 5
  0 of 5 complete

scratch directory removed: True
repository untouched: nothing outside traceback_lab_b40amygd was written at any point


## 🔑 Key takeaways — the sixty-second protocol

Print this. Tape it to the wall. Do it in this order, every time.

0. **Do not restart the kernel.** The halt destroyed nothing; restarting destroys everything you
   had loaded, including the failure you were about to read. Restart *after* the fix, never before.
1. **Bottom line first.** Exception **type**, then **message**. Type over message: the type is a
   fact, the message is prose.
2. **Is there more than one traceback?** If yes, scroll to the **top** one — that is the cause.
   *"During handling of the above exception"* means the handler is broken too.
3. **Bottom frame = where it stopped.** File, line, function. Read the **carets** before the line.
4. **Deepest frame that is yours = where you handed over bad input.** That is usually where the fix
   goes.
5. **Walk up asking one question per frame:** *"was this value already wrong when it arrived?"* The
   first frame that answers *no, and I spoiled it* is the bug.
6. **Measure, do not deduce.** Print the shapes, the dtypes, the lengths, `repr(columns)`,
   `module.__file__`. Confirm the diagnosis against the numbers in the message.
7. **Still stuck? Stop reading and bisect.** Halve the data; halve the history with
   `git bisect run`; halve the code by deleting until it works.
8. **Before you leave, make it loud.** Add the `assert` that would have turned this into a traceback
   an hour earlier, and an `add_note` that would have said which file.

And the two sentences that carry the rest of the lesson:

> **Where it broke is not where it went wrong.**
>
> **The silent failures are the expensive ones. If it did not raise, that is not good news.**

## 💬 Discuss

Argue every one of these from the numbers this notebook actually printed — the `(714,)` vs `(891,)`
shapes, the 177 missing ages, the 2 missing ports at rows 61 and 829, the 186 usable values out
of 891 in Task 5, the 1 leaked row in the 80/20 split, the 12 falsely-accused cells across 8 notebooks, and
the probe counts from both bisections.

1. **Section 2 offered three defensible repairs and two of them printed the same number here.** A
   teammate says "they agree, so it does not matter which we use." Say why that reasoning is unsafe,
   using the third variant printed in the same cell. Then state which repair *you* would put in a
   pull request and what you would write in the commit message so the next person knows it was a
   choice.

2. **The four silent failures in Section 5 produced no traceback at all.** Rank them by how much
   money you think each could cost before anyone notices, and defend your ranking. Then pick the one
   you rank most dangerous and write the single `assert` that would have caught it — and say honestly
   where you would have had to already suspect the problem in order to write that line.

3. **Section 8's tooling bug falsely accused 12 cells across 8 notebooks.** Suppose you inherit that
   checker with those 12 failures already sitting in a baseline file marked "known issues". Describe
   what happens to the team over the following six months, and what you would do in week one
   instead. Is a growing list of accepted failures ever the right engineering answer?

4. **`git bisect run` found the guilty commit in a handful of test runs.** It only works if the test
   is deterministic. Describe a realistic way a machine-learning test could be flaky, say what
   `git bisect` would conclude, and design the smallest change to the test that makes bisection
   trustworthy again.

5. **Barik et al. measured 13%–25% of task time spent reading error messages.** That study used
   Java in Eclipse with 56 students. Say specifically why you should be cautious about carrying that
   figure over to Python in Jupyter, and describe how you would measure the same thing in this
   classroom in one afternoon.

6. **Section 1a is the only cell in this diploma that stops your kernel.** Everywhere else, including
   the rest of this notebook, failure is caught and printed. Argue the number. One halting cell
   teaches the habit; a hundred would mean a student whose kernel dies at cell 3 loses the whole
   session — in a second language, with one instructor for twenty people. Where would you put the
   line, and which three lessons specifically would you allow to break? Say what makes those three
   different from the rest, and what evidence would settle the question rather than decide it by
   taste.

## ⚠️ Where this breaks

- **This whole strand has no outcome evidence behind it, and that must be said plainly.** There is no
  trial showing that teaching tracebacks, shells, environments or `git bisect` makes graduates
  better. The strand is here on a **prerequisite argument**, not a proven one: every other
  intervention worth doing — peer review of real work, written feedback inside real artefacts,
  handing a student a repository and expecting them to be useful — presupposes that the student can
  operate the tools. That is an argument about necessity, and it is weaker than an argument from
  measured effect. Treat this lesson as scaffolding for what is evidenced, not as an evidenced
  intervention in itself.

- **The halt in Section 1a is authored, and that is a limit as much as it is the point.** You were
  told it was coming, the cause is explained on the next page, the file is 59 KB and the failure
  arrives instantly. At work you will meet a halt you did not expect, in code you did not write, and
  re-triggering it may cost forty minutes on a machine you share with other people. The reading
  technique is identical; the patience it demands is not, and nothing here rehearses that.

- **The tag that makes that halt verifiable is a tool convention, not a language feature.** The
  halting cell carries `raises-exception` in its metadata. nbconvert's documentation says errors
  "can be allowed with a `raises-exception` tag on a single cell, or the `allow_errors` or
  `allow_error_names` configurable options for all cells" — so the repository's checker executes this
  notebook to the end instead of calling the failure a defect, and it also reports a *failure* if
  that cell ever stops raising. JupyterLab and Colab do not read the tag at all: there, *Run All*
  simply stops, which is what you want as a student. But run this file anywhere that allows errors
  globally and the halt becomes invisible, with the cells after it failing for reasons that look
  unrelated to it. We have not tested every editor a student might use and cannot promise how yours
  behaves — check it once, on this notebook, before you trust it on your own work.

- **Tracebacks in a notebook are the easy case.** The kernel survives, the traceback is coloured,
  and the variables are still alive. In production you will meet failures with no traceback at all:
  a segmentation fault in a C extension, an OOM kill by the operating system, a container that
  vanishes, a process that hangs forever without failing. None of this lesson's technique applies
  to those. They need logs, `dmesg`, `py-spy`, `faulthandler` and system tools.

- **`pdb` cannot help with the failures that matter most in ML.** Post-mortem debugging assumes a
  crash. A model that trains to an impressive accuracy on a leaked test set never crashes, and the
  leak is the only thing wrong with it. The quiet four in
  Section 5 are a small sample of a very large family, and the antidote is assertions, data
  validation (`pandera`, Great Expectations) and review — not a debugger.

- **The scripted `pdb` session here is a teaching device.** Real debugging is a conversation: you
  look, you form a hypothesis, you ask the next question. Feeding a fixed command list captures the
  *transcript* but not the *thinking*, and the thinking is the skill. Do Task 4 at a real prompt in
  a terminal, not only in this notebook.

- **Bisection needs monotonicity, and real bugs are often not monotone.** A failure that depends on
  two interacting rows, or on ordering, or on a random seed, will make bisection point confidently
  at an innocent commit or an innocent row. Bisection also cannot find a bug that was always there
  and only became *visible* last month.

- **`SettingWithCopyWarning` is version-dependent.** This notebook prints what pandas actually did on
  the installed version. Under pandas Copy-on-Write — opt-in in 2.x and the default in 3.0 — the
  behaviour and the warning both change. The lesson (an assignment can silently go nowhere) survives
  the version change; the exact printout does not. Always trust the output your own environment
  produces over anything written about it, including this paragraph.

- **The exact traceback text is not stable across Python versions.** The carets are Python 3.11 and
  later (PEP 657); older interpreters name only a line. Some messages have been reworded between
  releases. Read the structure, which is stable, rather than memorising strings, which are not.

- **A one-row reproduction can mislead.** Section 7a shrank the failure to a single row, which is
  usually ideal — but if the real defect is an interaction between rows, the minimal reproduction you
  build will be a *different, simpler* bug that happens to share a symptom.

## 📚 References

1. Galindo Salgado, P., Taskaya, B., & Askar, A. (2021). *PEP 657 – Include Fine Grained Error
   Locations in Tracebacks*. Status: Final; Python 3.11. <https://peps.python.org/pep-0657/>
2. Hatfield-Dodds, Z. (2021). *PEP 678 – Enriching Exceptions with Notes*. Status: Final;
   Python 3.11. <https://peps.python.org/pep-0678/>
3. Barik, T., Smith, J., Lubick, K., Holmes, E., Feng, J., Murphy-Hill, E., & Parnin, C. (2017).
   *Do Developers Read Compiler Error Messages?* In Proceedings of the 39th International Conference
   on Software Engineering (ICSE '17). DOI: 10.1109/ICSE.2017.59.
   <https://dl.acm.org/doi/10.1109/ICSE.2017.59>
4. Yee, K.-P. (2005). *PEP 3134 – Exception Chaining and Embedded Tracebacks*. Status: Final;
   Python 3.0. The source of `__cause__`, `__context__`, `__traceback__` and the two
   "above exception" sentences. <https://peps.python.org/pep-3134/>
5. Python Software Foundation. *`pdb` — The Python Debugger*, standard library documentation.
   <https://docs.python.org/3/library/pdb.html>
6. Python Software Foundation. *`traceback` — Print or retrieve a stack traceback*, standard library
   documentation. <https://docs.python.org/3/library/traceback.html>
7. Git project. *`git-bisect` — Use binary search to find the commit that introduced a bug*.
   <https://git-scm.com/docs/git-bisect>
8. pandas development team. *Returning a view versus a copy* and *Copy-on-Write*, pandas user guide.
   <https://pandas.pydata.org/docs/user_guide/indexing.html#returning-a-view-versus-a-copy>
9. Zeller, A. (2009). *Why Programs Fail: A Guide to Systematic Debugging* (2nd ed.). Morgan
   Kaufmann. The standard scholarly treatment of bisection, delta debugging, and hypothesis-driven
   defect isolation.
10. Samuel, S., & Mietchen, D. (2022). *Computational reproducibility of Jupyter notebooks from
    biomedical publications*. arXiv:2209.04308. The source of the 396-of-2,684 and 245-of-2,684
    figures quoted in Section 1a. <https://arxiv.org/abs/2209.04308>
11. Jupyter Project. *nbconvert — Configuration options*: `ExecutePreprocessor.allow_errors` and the
    `raises-exception` cell tag. This is what lets Section 1a halt for you and still be checked by
    this repository. <https://nbconvert.readthedocs.io/en/latest/config_options.html>